Enable GPU

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable a GPU in Runtime > Change runtime type.")

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


Mount Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Verify Adaptors

In [3]:
from pathlib import Path

ADAPTER_DIR = Path(
    "/content/drive/MyDrive/Mental_Health_Chatbot/"
    "Qwen_lora/final_adapter"
)

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

for name in required_files:
    print(name, "FOUND" if (ADAPTER_DIR / name).is_file() else "MISSING")

if not all((ADAPTER_DIR / name).is_file() for name in required_files):
    raise FileNotFoundError(
        f"Your saved adapter is incomplete or the path is incorrect: {ADAPTER_DIR}"
    )

print("Adapter folder verified:", ADAPTER_DIR)

adapter_config.json FOUND
adapter_model.safetensors FOUND
Adapter folder verified: /content/drive/MyDrive/Mental_Health_Chatbot/Qwen_lora/final_adapter


Install the required packages

In [4]:
%pip install -q streamlit transformers peft accelerate bitsandbytes

Create the Streamlit chatbot

In [5]:
%%writefile /content/app.py
from pathlib import Path

import torch
import streamlit as st
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

ADAPTER_DIR = Path(
    "/content/drive/MyDrive/Mental_Health_Chatbot/"
    "Qwen_lora/final_adapter"
)

SYSTEM_MESSAGE = """
You are a supportive mental-health information assistant.
Respond with empathy, respect, and clear language.
Do not diagnose medical conditions or prescribe medication.
Do not claim to replace a qualified mental-health professional.
Offer general coping strategies and encourage professional support
when appropriate.
"""

CRISIS_MESSAGE = (
    "I'm sorry you're going through such a difficult time. Your safety "
    "matters. If you may act on thoughts of suicide or are in immediate "
    "danger in the UK, call 999 or go to A&E. You can call Samaritans "
    "free on 116 123 at any time. If possible, contact someone you trust "
    "and ask them to stay with you. This chatbot cannot provide emergency care."
)

st.set_page_config(
    page_title="MindCare Chat",
    page_icon="💬",
    layout="centered",
)

st.title("💬 MindCare Chat")
st.caption(
    "A research prototype for supportive mental-health conversations. "
    "It is not a therapist, diagnostic tool, or emergency service."
)

@st.cache_resource
def load_model(adapter_path):
    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU not available. Select Runtime > Change runtime type > GPU."
        )

    adapter_path = Path(adapter_path)

    if not (adapter_path / "adapter_config.json").is_file():
        raise FileNotFoundError(
            f"Adapter configuration not found: {adapter_path}"
        )

    if not (adapter_path / "adapter_model.safetensors").is_file():
        raise FileNotFoundError(
            f"Adapter weights not found: {adapter_path}"
        )

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        torch_dtype=dtype,
        device_map="auto",
    )

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
    )
    model.eval()

    return tokenizer, model


try:
    with st.spinner("Loading your fine-tuned Qwen assistant..."):
        tokenizer, model = load_model(str(ADAPTER_DIR))
    st.success("Your assistant is ready.")
except Exception as exc:
    st.error(f"Could not load the model: {exc}")
    st.stop()


if "messages" not in st.session_state:
    st.session_state.messages = []

# Display previous messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

prompt = st.chat_input("What would you like to talk about?")

if prompt:
    st.session_state.messages.append(
        {"role": "user", "content": prompt}
    )

    with st.chat_message("user"):
        st.markdown(prompt)

    lower_prompt = prompt.lower()
    crisis_terms = [
        "kill myself",
        "suicide",
        "end my life",
        "want to die",
        "self harm",
        "self-harm",
        "hurt myself",
    ]

    if any(term in lower_prompt for term in crisis_terms):
        answer = CRISIS_MESSAGE
    else:
        # Keep recent context to support a multi-turn conversation.
        recent_messages = st.session_state.messages[-10:]

        chat_messages = [
            {"role": "system", "content": SYSTEM_MESSAGE}
        ] + recent_messages

        try:
            inputs = tokenizer.apply_chat_template(
                chat_messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)

            with st.spinner("Thinking..."):
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=220,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        repetition_penalty=1.1,
                        pad_token_id=tokenizer.pad_token_id,
                    )

            new_tokens = outputs[
                0, inputs["input_ids"].shape[-1]:
            ]

            answer = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            ).strip()

            if not answer:
                answer = "I couldn't generate a response. Please try again."

        except Exception as exc:
            answer = f"Response generation failed: {exc}"

    with st.chat_message("assistant"):
        st.markdown(answer)

    st.session_state.messages.append(
        {"role": "assistant", "content": answer}
    )

if st.button("Clear conversation"):
    st.session_state.messages = []
    st.rerun()

Overwriting /content/app.py


Start Streamlit

In [14]:
import subprocess
import time
from pathlib import Path

# Stop an existing server using port 8501, if any
subprocess.run(
    ["bash", "-lc", "fuser -k 8501/tcp"],
    capture_output=True,
    text=True,
)

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print("Streamlit process running:", streamlit_process.poll() is None)
print(Path("/content/streamlit.log").read_text()[-5000:])

Streamlit process running: True


2026-10-10 09:49:44.773 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.50.184.233:8501




In [15]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sat, 10 Oct 2026 09:49:50 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Sat, 10 Oct 2026 09:04:58 GMT
etag: "8a6cb0a2f012f095b1bdffa95706dafc"
cache-control: no-cache



Install LocalTunnel:

In [8]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [9]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://free-teams-move.loca.lt
^C


Try Cloudflare tunnel

In [12]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
! /content/cloudflared tunnel --url http://127.0.0.1:8501

2026-10-10T09:50:10Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-10T09:50:10Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-10T09:50:15Z INF +--------------------------------------------------------------------------------------------+
2026-10-10T09:50:15Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-10T09:50:15Z INF |  https://sword-parenting-centers-quality.trycloudflare